In [ ]:
import pandas as pd
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from pathlib import Path

DATA_PATH = Path('../data/fer2013.csv')

def load_fer2013(path):
    df = pd.read_csv(path)
    pixels = df['pixels'].tolist()
    width = 48
    images = np.array([np.fromstring(pixel_sequence, sep=' ', dtype='float32').reshape((width, width)) for pixel_sequence in pixels])
    labels = df['emotion'].values
    images = images[..., np.newaxis] / 255.0
    return images, labels

x, y = load_fer2013(DATA_PATH)
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', x_train.shape, 'Validation:', x_val.shape)

In [ ]:
def build_emotion_cnn(input_shape=(48, 48, 1), num_classes=7):
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_emotion_cnn()
model.summary()

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=20,
    batch_size=64,
)